# Does the City's cycling plan cover the gaps?

Notebook 07 found 50 high-demand routes with almost no protected bike-lane coverage. The obvious follow-up: **are those routes already in the City's plans?**

Three plan datasets:
- **Near-term 2025–2027 Programmed Cycling Projects** — current three-year rolling plan (599 features, 546 km)
- **Near-term 2022–2024 Programmed Cycling Projects** — the program that just finished (332 features, 347 km; some already built & in the cycling-network feed)
- **10-Year Cycling Network Plan (2016)** — on-street (863 features) + trails (126 features)

For each plan, keep only features that represent real protected infrastructure (cycle tracks, bike lanes, multi-use trails, bridges) — drop "Signed and marked route", "Study or Design", "Quiet Street" etc. Then union all three into one "planned protected infrastructure" geometry, buffer by 50 m, and measure how much of each gap route falls inside.

If most gap routes are covered by existing plans, great — the data validates the City's priorities. If most are *not*, the data is pointing at a blind spot.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import duckdb
import geopandas as gpd
from shapely.geometry import LineString
import pydeck as pdk

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROC = Path.cwd().parent / 'data' / 'processed'
METRIC = 'EPSG:2952'

## 1. Load plans and filter to protected facility types

In [2]:
# 2025–2027 near-term programmed
p2527 = gpd.read_file(f'zip://{DATA_RAW}/programmed_2025_2027.zip').to_crs(METRIC)
# Drop 'Study or Design' (not committed). Keep everything else.
p2527 = p2527[~p2527['map_legend'].isin(['Study or Design'])].copy()

# 2022–2024 near-term programmed
p2224 = gpd.read_file(f'zip://{DATA_RAW}/programmed_2022_2024.zip').to_crs(METRIC)
p2224 = p2224[~p2224['Map_Legend'].isin(['Study'])].copy()

# 10-year trails (2016)
t10 = gpd.read_file(f'zip://{DATA_RAW}/plan_trails_10yr.zip').to_crs(METRIC)
t10 = t10[t10['Facil'].isin(['Trail', 'Bicycle Lane', 'Bike Lane or cycle track', 'Cycle Track', 'Bike Lane'])].copy()

# 10-year on-street (2016) — filter to protected/real infrastructure.
o10 = gpd.read_file(f'zip://{DATA_RAW}/plan_onstreet_10yr.zip').to_crs(METRIC)
PROTECTED_10 = {
    'Cycle tracks', 'Bike lanes', 'Buffered bike lanes', 'Boulevard multi-use trail',
    'Multi-use trail', 'Boulevard multi-use trail or cycle track', 'Bike lanes or cycle tracks',
    'Shoulder multi-use trail', 'Bridge', 'Tunnel',
    'Signed and marked route with contra-flow bike lane',  # contra-flow is a real lane
}
o10 = o10[o10['CN_FACILIT'].isin(PROTECTED_10)].copy()

print(f'Near-term 2025-2027 protected: {len(p2527)} features, {p2527.length.sum()/1000:.1f} km')
print(f'Near-term 2022-2024 protected: {len(p2224)} features, {p2224.length.sum()/1000:.1f} km')
print(f'10-year trails protected:      {len(t10)} features, {t10.length.sum()/1000:.1f} km')
print(f'10-year on-street protected:   {len(o10)} features, {o10.length.sum()/1000:.1f} km')

Near-term 2025-2027 protected: 381 features, 284.7 km
Near-term 2022-2024 protected: 201 features, 182.0 km
10-year trails protected:      103 features, 104.1 km
10-year on-street protected:   609 features, 374.1 km


## 2. Union all plans + buffer 50 m

In [3]:
all_plans = pd.concat([
    p2527[['geometry']].assign(source='near_2527'),
    p2224[['geometry']].assign(source='near_2224'),
    t10[['geometry']].assign(source='10yr_trails'),
    o10[['geometry']].assign(source='10yr_onstreet'),
], ignore_index=True)
all_plans = gpd.GeoDataFrame(all_plans, crs=METRIC)

BUFFER_M = 50
plans_union = all_plans.buffer(BUFFER_M).union_all()
print(f'Unioned buffer area: {plans_union.area/1e6:.1f} km²')

# Compare to built infra for context
built = gpd.read_file(DATA_RAW / 'cycling_network_4326.geojson')
PROTECTED_BUILT = {
    'Multi-Use Trail', 'Multi-Use Trail - Entrance', 'Multi-Use Trail - Boulevard',
    'Multi-Use Trail - Existing Connector', 'Multi-Use Trail - Connector',
    'Bike Lane', 'Bike Lane - Contraflow', 'Bike Lane - Buffered',
    'Cycle Track', 'Cycle Track - Contraflow', 'Bi-Directional Cycle Track',
}
built = built[built['INFRA_HIGHORDER'].isin(PROTECTED_BUILT)].to_crs(METRIC)
built_union = built.buffer(BUFFER_M).union_all()
print(f'Built protected buffer area:   {built_union.area/1e6:.1f} km²')

Unioned buffer area: 61.9 km²


Built protected buffer area:   59.0 km²


## 3. Recompute gap candidates and measure overlap with PLANS

In [4]:
# Recompute the gap candidates table from scratch (same logic as notebook 07).
stations = pd.read_parquet(DATA_RAW / 'stations.parquet')
stations['sid'] = pd.to_numeric(stations['station_id'], errors='coerce').astype('Int64')
stations = stations.dropna(subset=['sid']).set_index('sid')[['name', 'lat', 'lon', 'capacity']]

GLOB = str(DATA_RAW / 'ridership' / '2025' / '*.csv')
con = duckdb.connect()
con.execute(f"CREATE VIEW trips AS SELECT * FROM read_csv_auto('{GLOB}', header=True, ignore_errors=true, union_by_name=true)")
edges = con.execute('''
    SELECT LEAST(Start_Station_Id, End_Station_Id) AS a,
           GREATEST(Start_Station_Id, End_Station_Id) AS b,
           COUNT(*) AS trips
    FROM trips
    WHERE Start_Station_Id IS NOT NULL AND End_Station_Id IS NOT NULL AND Start_Station_Id <> End_Station_Id
    GROUP BY 1, 2
''').fetchdf()
edges = edges.join(stations.add_prefix('a_'), on='a', how='inner') \
             .join(stations.add_prefix('b_'), on='b', how='inner')

def hav(la1, lo1, la2, lo2):
    R = 6371.0
    la1, la2 = np.radians(la1), np.radians(la2)
    dlat = la2 - la1; dlon = np.radians(lo2 - lo1)
    a = np.sin(dlat/2)**2 + np.cos(la1) * np.cos(la2) * np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))
edges['dist_km'] = hav(edges['a_lat'], edges['a_lon'], edges['b_lat'], edges['b_lon']).clip(lower=0.05)

fit = edges[(edges['a_capacity']>0) & (edges['b_capacity']>0) & (edges['trips']>0)]
slope, intercept = np.polyfit(np.log(fit['dist_km']), np.log(fit['trips']) - np.log(fit['a_capacity']) - np.log(fit['b_capacity']), 1)
alpha, K = -slope, np.exp(intercept)
edges['expected'] = K * edges['a_capacity'].astype(float) * edges['b_capacity'].astype(float) / (edges['dist_km']**alpha)
edges['affinity'] = edges['trips'] / edges['expected']
cands = edges[(edges['trips']>=300) & (edges['a_capacity']>0) & (edges['b_capacity']>0) & (edges['dist_km']>=0.3) & np.isfinite(edges['affinity'])].copy()
print(f'{len(cands)} candidate pairs (recomputed from scratch)')

3348 candidate pairs (recomputed from scratch)


In [5]:
# Compute lane_cov against BOTH built and plans.
lines = gpd.GeoSeries([LineString([(r.a_lon, r.a_lat), (r.b_lon, r.b_lat)]) for r in cands.itertuples(index=False)], crs='EPSG:4326').to_crs(METRIC)
cands = cands.copy()
cands['line_m'] = lines.length.values
cands['built_m'] = lines.intersection(built_union).length.values
cands['plan_m']  = lines.intersection(plans_union).length.values
cands['combined_m'] = lines.intersection(built_union.union(plans_union)).length.values
cands['built_cov']    = cands['built_m']    / cands['line_m']
cands['plan_cov']     = cands['plan_m']     / cands['line_m']
cands['combined_cov'] = cands['combined_m'] / cands['line_m']
print('Lane-coverage summary (share of straight-line within 50m of each):')
print(cands[['built_cov', 'plan_cov', 'combined_cov']].describe().round(3))

Lane-coverage summary (share of straight-line within 50m of each):
       built_cov  plan_cov  combined_cov
count   3348.000  3348.000      3348.000
mean       0.404     0.337         0.487
std        0.267     0.268         0.270
min        0.000     0.000         0.000
25%        0.203     0.141         0.278
50%        0.345     0.273         0.447
75%        0.546     0.481         0.667
max        1.000     1.000         1.000


## 4. Gap routes — does the plan cover them?

Gap = high demand + high affinity + low BUILT coverage. Now we add: what's the PLAN coverage?

In [6]:
gaps = cands[
    (cands['trips'] >= 400) &
    (cands['affinity'] >= cands['affinity'].quantile(0.75)) &
    (cands['built_cov'] < 0.20) &
    (cands['dist_km'] >= 0.5)
].copy()
gaps = gaps.sort_values('trips', ascending=False)
print(f'{len(gaps)} gap pairs (same as notebook 07)')

# How many are covered by plans?
print(f'\nOf the top-50 gaps:')
top50 = gaps.head(50)
print(f'  mean built coverage:    {top50["built_cov"].mean():.1%}')
print(f'  mean plan coverage:     {top50["plan_cov"].mean():.1%}')
print(f'  mean combined coverage: {top50["combined_cov"].mean():.1%}')
print(f'  gaps with ≥ 50% plan coverage: {(top50["plan_cov"] >= 0.5).sum()} / 50')
print(f'  gaps with ≥ 30% plan coverage: {(top50["plan_cov"] >= 0.3).sum()} / 50')
print(f'  gaps with <  10% plan coverage (still not in plans): {(top50["plan_cov"] < 0.1).sum()} / 50')

97 gap pairs (same as notebook 07)

Of the top-50 gaps:
  mean built coverage:    12.0%
  mean plan coverage:     17.1%
  mean combined coverage: 22.7%
  gaps with ≥ 50% plan coverage: 3 / 50
  gaps with ≥ 30% plan coverage: 5 / 50
  gaps with <  10% plan coverage (still not in plans): 20 / 50


In [7]:
# The specific uncovered gaps — these are the ones neither built nor planned.
orphans = top50[top50['combined_cov'] < 0.15].sort_values('trips', ascending=False).copy()
print(f'\n{len(orphans)} "orphan" gaps — high demand, not built, not planned:')
out = orphans[['a_name', 'b_name', 'trips', 'dist_km', 'affinity', 'built_cov', 'plan_cov']].copy()
out['dist_km'] = out['dist_km'].round(2)
out['affinity'] = out['affinity'].round(1)
out['built_cov'] = (out['built_cov']*100).round(1).astype(str) + '%'
out['plan_cov']  = (out['plan_cov']*100).round(1).astype(str) + '%'
print(out.to_string(index=False))


15 "orphan" gaps — high demand, not built, not planned:
                        a_name                                    b_name  trips  dist_km  affinity built_cov plan_cov
     Windsor St / Newcastle St                         Grand Avenue Park   3229     0.92      22.6      0.0%     0.0%
         College St / Major St            Robert St / Bloor St W - SMART   1923     0.93      35.9     13.3%    13.7%
         College St / Huron St                      Soho St / Queen St W   1268     1.01      27.1     13.7%     5.7%
          Soho St / Queen St W          Baldwin St / Spadina Ave - SMART   1159     0.70      12.3      4.5%     2.6%
Marilyn Bell Park Tennis Court Humber Bay Shores Park / Marine Parade Dr    877     3.26      19.1      3.7%     0.0%
      Bloor St W / Dundas St W          Roncesvalles Ave / Fermanagh Ave    827     1.14      20.4      8.5%     3.6%
   Humber Bay Shores Park East            Marilyn Bell Park Tennis Court    796     2.82      37.7      8.2%     0.0%

## 5. Gap-vs-plans map

Four-colour overlay:
- 🟢 green: existing protected infrastructure
- 🟡 amber: planned (near-term or 10-year)
- 🔴 red: gap routes covered by plans
- 🟣 magenta: orphan gap routes — high demand, not built, not planned

In [8]:
# Bike-lane layer (built)
built_wgs = built.to_crs('EPSG:4326')
built_paths = []
for geom in built_wgs.geometry:
    if geom.geom_type == 'LineString':
        built_paths.append({'path': [list(c) for c in geom.coords]})
    elif geom.geom_type == 'MultiLineString':
        for line in geom.geoms:
            built_paths.append({'path': [list(c) for c in line.coords]})

# Plan layer (union of all plans)
plans_wgs = all_plans.to_crs('EPSG:4326')
plan_paths = []
for geom in plans_wgs.geometry:
    if geom.geom_type == 'LineString':
        plan_paths.append({'path': [list(c) for c in geom.coords]})
    elif geom.geom_type == 'MultiLineString':
        for line in geom.geoms:
            plan_paths.append({'path': [list(c) for c in line.coords]})

built_layer = pdk.Layer('PathLayer', data=built_paths, get_path='path',
                         get_color=[30, 140, 70, 180], get_width=2.5, width_min_pixels=1.5)
plan_layer  = pdk.Layer('PathLayer', data=plan_paths,  get_path='path',
                         get_color=[245, 185, 50, 170], get_width=2.2, width_min_pixels=1.3)

# Gap arcs — split into "planned" (covered) and "orphan" (not planned)
gaps_planned = top50[top50['plan_cov'] >= 0.3].copy()
gaps_orphan  = top50[top50['plan_cov'] < 0.3].copy()

for g in (gaps_planned, gaps_orphan):
    g['width'] = np.clip(np.log1p(g['trips'].astype(float)) * 1.1, 2.0, 10.0)
    g['pair'] = g['a_name'].astype(str) + ' ↔ ' + g['b_name'].astype(str)
    g['aff_r'] = g['affinity'].round(1)
    g['trips_k'] = (g['trips']/1000).round(2)
    g['built_r'] = (g['built_cov']*100).round(1)
    g['plan_r']  = (g['plan_cov']*100).round(1)

planned_arcs = pdk.Layer(
    'ArcLayer',
    data=gaps_planned[['a_lat', 'a_lon', 'b_lat', 'b_lon', 'width', 'pair', 'aff_r', 'trips_k', 'built_r', 'plan_r']],
    get_source_position=['a_lon', 'a_lat'], get_target_position=['b_lon', 'b_lat'],
    get_width='width',
    get_source_color=[220, 70, 70, 235], get_target_color=[220, 70, 70, 235],
    get_height=0.45, pickable=True, auto_highlight=True,
)
orphan_arcs = pdk.Layer(
    'ArcLayer',
    data=gaps_orphan[['a_lat', 'a_lon', 'b_lat', 'b_lon', 'width', 'pair', 'aff_r', 'trips_k', 'built_r', 'plan_r']],
    get_source_position=['a_lon', 'a_lat'], get_target_position=['b_lon', 'b_lat'],
    get_width='width',
    get_source_color=[200, 60, 210, 240], get_target_color=[200, 60, 210, 240],
    get_height=0.55, pickable=True, auto_highlight=True,
)

view = pdk.ViewState(latitude=43.655, longitude=-79.420, zoom=12.3, pitch=50, bearing=-10)
deck = pdk.Deck(
    layers=[built_layer, plan_layer, planned_arcs, orphan_arcs],
    initial_view_state=view,
    map_style='light',
    tooltip={'text': '{pair}\n{trips_k}k trips · affinity {aff_r}×\nbuilt coverage: {built_r}%   plan coverage: {plan_r}%'},
)
out = DATA_PROC / 'gap_vs_plans_map.html'
deck.to_html(str(out), notebook_display=False)
print(f'Wrote: {out}')
print(f'Planned arcs (red): {len(gaps_planned)}')
print(f'Orphan arcs (magenta): {len(gaps_orphan)}')

Wrote: /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/gap_vs_plans_map.html
Planned arcs (red): 5
Orphan arcs (magenta): 45


In [9]:
# Save orphan table for the story page + memory
orphan_out = gaps_orphan[['a_name', 'b_name', 'trips', 'dist_km', 'affinity', 'built_cov', 'plan_cov', 'combined_cov']].copy()
orphan_out['dist_km'] = orphan_out['dist_km'].round(2)
orphan_out['affinity'] = orphan_out['affinity'].round(1)
for c in ['built_cov', 'plan_cov', 'combined_cov']:
    orphan_out[c] = (orphan_out[c]*100).round(1)
orphan_out.to_csv(DATA_PROC / 'gap_orphans_table.csv', index=False)
print(f'Wrote {DATA_PROC}/gap_orphans_table.csv')
orphan_out.head(15)

Wrote /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/gap_orphans_table.csv


,a_name,b_name,trips,dist_km,affinity,built_cov,plan_cov,combined_cov
141118,Windsor St / Newcastle St,Grand Avenue Park,3229,0.92,22.6,0.0,0.0,0.0
73302,College St / Major St,Robert St / Bloor St W - SMART,1923,0.93,35.9,13.3,13.7,13.8
179914,Windsor St / Newcastle St,36 Park Lawn Rd,1781,1.29,32.7,0.0,16.2,16.2
40840,Union Station,Front St W / Blue Jays Way,1755,0.85,18.8,11.8,22.5,23.1
40580,Dundas St E / Regent Park Blvd,Yonge St / Dundas Sq,1656,1.60,23.7,17.2,7.2,18.8
77386,Dundas St W / Yonge St,Dundas St E / Regent Park Blvd,1605,1.69,25.0,16.6,11.9,22.9
107780,Union Station,Front St W / Spadina Ave,1580,1.25,15.2,16.0,22.9,23.6
141340,King St W / Portland St,Simcoe St / King St W,1486,1.23,25.0,12.8,22.3,22.3
210777,College St / Huron St,Soho St / Queen St W,1268,1.01,27.1,13.7,5.7,14.1
181995,Soho St / Queen St W,Baldwin St / Spadina Ave - SMART,1159,0.70,12.3,4.5,2.6,4.8
